# CommuteCast — Stage 2: Data Preprocessing & Exploratory Statistical Analysis

## Unit-1 Conceptual Foundations:
- **Descriptive Statistics**: Sample Mean, Sample Variance, Standard Deviation, Quartiles (Q25, Q50, Q75), and IQR.
- **Empirical Expectation**: Using the sample mean $\bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$ as the empirical estimator of expected traffic volume $\hat{E}[X]$ via the Law of Large Numbers.
- **Sample Covariance**: Quantifying joint linear variability between traffic volume and environmental features without confusing correlation/covariance with causation.
- **Quantile-Based Congestion**: Objective segmentation of traffic flow into *Low*, *Medium*, and *High* congestion regimes.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure presentation styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'figure.titlesize': 14,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))
from src.preprocessing import preprocess_traffic_data, load_raw_data
from src.statistics import compute_traffic_volume_statistics, compute_covariances, compute_congestion_distribution
from src.visualizations import (
    plot_traffic_volume_histogram,
    plot_avg_traffic_by_hour,
    plot_congestion_distribution,
    plot_traffic_rain_vs_norain,
    plot_hourly_traffic_with_observed_averages,
    generate_executive_dashboard
)

## 1. Load Data & Execute Preprocessing Pipeline

Key data-quality actions:
1. **Deduplication**: 17 exact duplicate rows removed (preserving first instance).
2. **Invalid Temperature (temp == 0 K)**: 10 records of absolute zero (sensor dropouts) set to `NaN` for temperature analysis while preserving valid traffic counts.
3. **Extreme Rainfall (9831.3 mm)**: Physically impossible gauge anomaly investigated. Row preserved (traffic count 5,535 is valid), binary `rain = 1` preserved, and numerical outlier set to `NaN` for robust covariance.
4. **Holiday String 'None'**: Preserved as literal string 'None' and mapped to `is_holiday = 0`.

In [ ]:
df = preprocess_traffic_data(raw_path='../data/Metro_Interstate_Traffic_Volume.csv', save_output=False)
print(f'Cleaned dataset shape: {df.shape}')
df[['date_time', 'hour', 'is_weekend', 'temp_celsius', 'rain', 'is_holiday', 'traffic_volume', 'congestion']].head()

## 2. Descriptive Statistics & Empirical Expectation

In probability theory, the **expected value** $E[X]$ is the theoretical probability-weighted average:
$$E[X] = \int_{-\infty}^{\infty} x \cdot f(x) \, dx$$

In an empirical study where the true population distribution $f(x)$ is unknown, the **sample mean** $\bar{x}$ serves as the empirical unbiased estimator:
$$\hat{E}[X] = \bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$$

By the **Weak Law of Large Numbers (WLLN)**, as $n \to \infty$, $\bar{x} \xrightarrow{P} E[X]$.

In [ ]:
stats = compute_traffic_volume_statistics(df)
for k, v in stats.items():
    print(f'{k:25s}: {v:,.2f}' if isinstance(v, float) else f'{k:25s}: {v:,}')

## 3. Sample Covariance Analysis & Causation Evaluation

Sample covariance formula:
$$Cov(X, Y) = \frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})$$

> **Unit-1 Core Principle: Covariance is NOT Causation**
>
> A positive covariance indicates that two variables tend to deviate in the same direction relative to their means, but does **not** demonstrate that one causes the other. For instance, `temp_celsius` has a positive covariance with `traffic_volume` because both peak during the daytime and summer, confounded by diurnal and seasonal schedules.

In [ ]:
covs = compute_covariances(df)
for k, v in covs.items():
    print(f'{k:30s}: {v:,.4f}' if isinstance(v, float) else f'{k:30s}: {v:,}')

## 4. Quantile-Based Congestion Classification

Categories are defined objectively using empirical quartiles:
- **Low**: $\text{traffic\_volume} \le Q_{25}$ ($1,192.5$ veh/hr)
- **Medium**: $Q_{25} < \text{traffic\_volume} \le Q_{75}$ ($(1,192.5, 4,933.0]$ veh/hr)
- **High**: $\text{traffic\_volume} > Q_{75}$ ($> 4,933.0$ veh/hr)

In [ ]:
congestion_df = compute_congestion_distribution(df)
display(congestion_df)

## 5. Visualizations for College Presentation

In [ ]:
# 1. Traffic Volume Histogram
fig1 = plot_traffic_volume_histogram(df)
plt.show()

In [ ]:
# 2. Average Traffic Volume by Hour
fig2 = plot_avg_traffic_by_hour(df)
plt.show()

In [ ]:
# 3. Congestion Category Distribution
fig3 = plot_congestion_distribution(df)
plt.show()

In [ ]:
# 4. Traffic Volume: Rain vs No Rain
fig4 = plot_traffic_rain_vs_norain(df)
plt.show()

In [ ]:
# 5. Traffic Volume by Hour with Observed Averages
fig5 = plot_hourly_traffic_with_observed_averages(df)
plt.show()

In [ ]:
# Combined Executive Dashboard
fig_dash = generate_executive_dashboard(df)
plt.show()